# Fintech Statement Engine: Comprehensive Analysis

This notebook provides a unified engine for analyzing bank statement data. It combines two powerful analysis layers:
1. **Transactional Rhythm**: Analyzing individual transfer consistency and timing.
2. **Monthly Stability (Rollup)**: Evaluating income capacity and stability across monthly windows using KMeans clustering and expert rules.

---

### Step 1: Setup & Configuration
We initialize the environment with a premium dark theme and unified data processing libraries.

In [ ]:
import pandas as pd, numpy as np, warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from IPython.display import HTML, display
warnings.filterwarnings('ignore')

# ══ CONFIGURATION ════════════════════════════════════════════════════════════
ACCOUNT_ID = 2  # Change this to analyze different accounts
CSV_PATH = 'credit_transactions.csv'

# ══ STYLING (Premium Dark Theme) ══════════════════════════════════════════════
plt.style.use('dark_background')
BG, PAN, GR, TX = '#0d1117', '#161b22', '#21262d', '#e6edf3'
COLORS = {
    'Stable Salary': '#58a6ff', 
    'B2C SME': '#f97583', 
    'B2B SME': '#3fb950', 
    'Other Business Income': '#d2a8ff', 
    'Ad-Hoc / Noise': '#6e7681'
}
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': PAN, 'axes.edgecolor': GR,
    'grid.color': GR, 'grid.linewidth': 0.5, 'text.color': TX,
    'axes.labelcolor': TX, 'xtick.color': TX, 'ytick.color': TX,
    'font.size': 11, 'axes.titlesize': 14, 'axes.titleweight': 'bold'
})
sns.set_palette(list(COLORS.values()))

def fmt_currency(x, _=None):
    if abs(x) >= 1e9: return f'{x/1e9:.1f}B'
    if abs(x) >= 1e6: return f'{x/1e6:.1f}M'
    if abs(x) >= 1e3: return f'{x/1e3:.0f}K'
    return str(int(x))

print(f"Setup complete. Targeting Account: {ACCOUNT_ID}")

### Step 2: Core Analysis Logic
We define the functions for both Transactional and Monthly analysis layers.

In [ ]:
def load_data(path):
    df = pd.read_csv(path)
    df['transaction_date'] = pd.to_datetime(df['transaction_date'], utc=True).dt.tz_convert('Asia/Ulaanbaatar')
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
    df['sender'] = df['sender'].astype(str).str.strip()
    df = df.dropna(subset=['amount', 'transaction_date', 'sender'])
    df = df[df['amount'] > 0]
    return df.sort_values(['account_id', 'sender', 'transaction_date']).reset_index(drop=True)

# --- LAYER 1: Transactional Rhythm Features ---
def engineer_transactional_features(df):
    rows = []
    for (acct, sender), g in df.groupby(['account_id', 'sender'], sort=False):
        g = g.sort_values('transaction_date')
        a = g['amount'].values; d = g['transaction_date']; n = len(g)
        mu = a.mean(); sd = a.std(ddof=1) if n > 1 else 0.0
        if n > 1:
            dl = d.diff().dt.total_seconds().dropna() / 86400
            mi, si = dl.mean(), (dl.std(ddof=1) if len(dl) > 1 else 0.0)
        else: mi = si = np.nan
        dow = d.dt.dayofweek; hr = d.dt.hour
        rows.append({
            'account_id': acct, 'sender': sender, 'N': n, 'total_amount': a.sum(),
            'mean_amount': mu, 'cv_amount': (sd / mu if mu > 0 else 0),
            'mean_interval': mi, 'std_interval': si,
            'weekend_pct': (dow >= 5).mean() * 100,
            'after_hours_pct': ((hr < 9) | (hr >= 18)).mean() * 100
        })
    return pd.DataFrame(rows)

def classify_transactional(r):
    SAL_SI, SAL_CV, NOISE_N = 3.0, 0.15, 2
    n, cv, si = r['N'], r['cv_amount'], r['std_interval']
    wk, ah = r['weekend_pct'], r['after_hours_pct']
    
    if n <= NOISE_N: return 'Ad-Hoc / Noise'
    if not np.isnan(si) and si < SAL_SI and cv < SAL_CV: return 'Stable Salary'
    if n >= 4 and wk >= 25 and ah >= 20: return 'B2C SME'
    if n >= 4 and cv > 0.30 and wk <= 10 and ah <= 10: return 'B2B SME'
    return 'Other Business Income'

# --- LAYER 2: Monthly Rollup & Clustering ---
def build_monthly_features(df):
    df = df.copy()
    df['month'] = df['transaction_date'].dt.to_period('M').astype(str)
    
    monthly_df = df.groupby(['account_id', 'sender', 'month'], as_index=False).agg(
        monthly_total=('amount', 'sum'),
        txns_in_month=('amount', 'size')
    )
    
    # Stability Metrics
    features = monthly_df.groupby(['account_id', 'sender']).agg(
        total_txns=('txns_in_month', 'sum'),
        avg_monthly_amt=('monthly_total', 'mean'),
        active_months=('month', 'nunique'),
        monthly_std=('monthly_total', 'std'),
        monthly_median=('monthly_total', 'median'),
        avg_txns_per_month=('txns_in_month', 'mean')
    ).reset_index()
    
    features['monthly_cv'] = features['monthly_std'].fillna(0) / features['avg_monthly_amt'].clip(lower=1)
    return features

def apply_clustering(features_df):
    if len(features_df) < 3: return features_df # Too few for clustering
    
    cols = ['avg_monthly_amt', 'active_months', 'monthly_cv']
    scaler = StandardScaler()
    X = scaler.fit_transform(features_df[cols].fillna(0))
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
    features_df['cluster'] = kmeans.fit_predict(X)
    return features_df

### Step 3: Execution & Analysis
We load the data and run both analysis layers to generate the results.

In [ ]:
raw_df = load_data(CSV_PATH)
acct_df = raw_df[raw_df['account_id'] == ACCOUNT_ID]

# Layer 1: Transactional
txn_features = engineer_transactional_features(acct_df)
txn_features['Category'] = txn_features.apply(classify_transactional, axis=1)

# Layer 2: Monthly
monthly_features = build_monthly_features(acct_df)
monthly_features = apply_clustering(monthly_features)

# Merge & Summary
summary = txn_features.merge(monthly_features[['sender', 'active_months', 'monthly_cv']], on='sender')
print(f"Analyzed {len(summary)} unique senders for Account {ACCOUNT_ID}.")
summary[['sender', 'Category', 'total_amount', 'N', 'active_months', 'cv_amount']].sort_values('total_amount', ascending=False).head(10)

### Step 4: Premium Dashboard
The unified dashboard visualizing income breakdown, stability, and monthly trends.

In [ ]:
# ══ PREPARATION ══════════════════════════════════════════════════════════════
inflow_total = summary['total_amount'].sum()
salary_df = summary[summary['Category'] == 'Stable Salary']
salary_total = salary_df['total_amount'].sum()
stability_score = (salary_total / inflow_total * 100) if inflow_total > 0 else 0

# Timeline Data
acct_df['month'] = acct_df['transaction_date'].dt.to_period('M').astype(str)
timeline = acct_df.merge(summary[['sender', 'Category']], on='sender')
pivot_timeline = timeline.pivot_table(index='month', columns='Category', values='amount', aggfunc='sum', fill_value=0)

# ══ HTML DASHBOARD ══════════════════════════════════════════════════════════
header_html = f'<h3 style="color:{TX};margin:8px 0">Account {ACCOUNT_ID} · Comprehensive Dashboard</h3>'
cards_html = f'''
<div style="display:flex;gap:10px;flex-wrap:wrap;margin:10px 0">
    <div style="background:{PAN};border:1px solid {GR};border-top:3px solid {COLORS['Stable Salary']};border-radius:8px;padding:14px 18px;min-width:160px;flex:1">
        <div style="color:#8b949e;font-size:10px;text-transform:uppercase;letter-spacing:1px;margin-bottom:4px">Total Inflow</div>
        <div style="color:{COLORS['Stable Salary']};font-size:22px;font-weight:700">{fmt_currency(inflow_total)} MNT</div>
        <div style="color:#6e7681;font-size:10px;margin-top:3px">Total lifecycle inflow</div>
    </div>
    <div style="background:{PAN};border:1px solid {GR};border-top:3px solid {COLORS['B2B SME']};border-radius:8px;padding:14px 18px;min-width:160px;flex:1">
        <div style="color:#8b949e;font-size:10px;text-transform:uppercase;letter-spacing:1px;margin-bottom:4px">Salary Inflow</div>
        <div style="color:{COLORS['B2B SME']};font-size:22px;font-weight:700">{fmt_currency(salary_total)} MNT</div>
        <div style="color:#6e7681;font-size:10px;margin-top:3px">From {len(salary_df)} stable streams</div>
    </div>
    <div style="background:{PAN};border:1px solid {GR};border-top:3px solid {COLORS['Other Business Income']};border-radius:8px;padding:14px 18px;min-width:160px;flex:1">
        <div style="color:#8b949e;font-size:10px;text-transform:uppercase;letter-spacing:1px;margin-bottom:4px">Stability Score</div>
        <div style="color:{COLORS['Other Business Income']};font-size:22px;font-weight:700">{stability_score:.1f}%</div>
        <div style="color:#6e7681;font-size:10px;margin-top:3px">Reliable income ratio</div>
    </div>
</div>
'''
display(HTML(header_html + cards_html))

# ══ VISUALIZATIONS ═══════════════════════════════════════════════════════════
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 2)

# 1. Income Breakdown (Donut)
ax1 = fig.add_subplot(gs[0, 0])
cat_dist = summary.groupby('Category')['total_amount'].sum()
colors = [COLORS.get(c, '#888') for c in cat_dist.index]
ax1.pie(cat_dist, labels=cat_dist.index, autopct='%1.1f%%', pctdistance=0.85, colors=colors, startangle=140, wedgeprops=dict(width=0.3))
ax1.set_title("Inflow Breakdown by Category")

# 2. Monthly Income Timeline
ax2 = fig.add_subplot(gs[0, 1])
pivot_timeline.plot(kind='bar', stacked=True, ax=ax2, color=[COLORS.get(c, '#888') for c in pivot_timeline.columns])
ax2.set_title("Monthly Inflow Composition")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(fmt_currency))
ax2.legend(loc='upper left', fontsize='small')

# 3. Stability Scatter (CV vs Total Amount)
ax3 = fig.add_subplot(gs[1, :])
for cat in summary['Category'].unique():
    subset = summary[summary['Category'] == cat]
    ax3.scatter(subset['total_amount'], subset['cv_amount'], label=cat, s=subset['N']*10, alpha=0.7, color=COLORS.get(cat, '#888'))
ax3.set_xscale('log')
ax3.set_title("Sender Stability: Coefficient of Variation vs. Total Volume")
ax3.set_xlabel("Total Amount (MNT) - Log Scale")
ax3.set_ylabel("CV (Lower = More Stable)")
ax3.xaxis.set_major_formatter(plt.FuncFormatter(fmt_currency))
ax3.legend()

plt.tight_layout()
plt.show()